# Train entity-encoder stack with PairHead + FiLM (bow+Ebi mix, T=6)

4-layer trainable stack (`PlanetEntityEncoder` → `CrossEntityAttention` → `DualRoleAttention` → `JointRoleAttention`) on top of 3 frozen L0 specialist encoders, capped by a **shared 2-layer trunk + L1-conditioned FiLM + 2 heads** producing `pair_logits` and `pair_frac`.

## Architecture (head side)

```
                       ctx_now (B, P, 256), source_joint, target_joint
                                  │
                       3× Linear(256 → 256) projections (d_pair = d_model = 256;
                       no down-projection), broadcast to (B, P, P, 6·256 = 1536)
                                  ▼
                       Linear(1536 → 256) → GELU → Linear(256 → 256) → GELU      ← shared trunk h
                                  │
                       FiLM cond = [L1_src ‖ L1_tgt ‖ pair_type_emb]
                       pair_type = source_type*9 + target_relation*3 + target_type (27 categories)
                       γ, β = MLP(cond), with γ=β=0 and α=1 at init
                       h_film = h + α · (γ · h + β)                              ← identity, trainable
                                  │
                         ┌────────┴────────┐
                         ▼                 ▼
                    pair_logits        pair_frac
                    (B,P,P) BCE        (B,P,P) MSE on sigmoid
                    pw=600             positive cells only
```

- `pair_logits`: per-cell source→target compatibility, masked by `pair_valid = mask[s] & mask[t] & (s != t)`.
- `pair_frac`: per-cell sigmoid → fraction of the source planet's ships sent to target; target is `pair_ships[s,t] / source_ships_before_launch[s]`, masked to positive cells only. Auto-skipped if `pair_ships` is missing from the batch.
- The former auxiliary heads (`source_act`, `target_aim`, `glob_act`) are gone; the runner only consumes `pair_logits` + `pair_frac`.

## MHA head counts (standardized to 8 heads, head_dim = 32 at d_model=256)

| Layer | block(s) | heads/block | n_layers |
|---|---|---|---|
| L1 PlanetEntityEncoder | 1× cross-attn (planet ← fleet) | 8 | 1 |
| L2 CrossEntityAttention | self-attn over (1+T·P) tokens | 8 | 2 |
| L3 DualRoleAttention | 2× cross-attn (src→tgt, tgt→src) | 8 | 1 each |
| L4 JointRoleAttention | self-attn over 2P tokens | 8 | 1 |

**Trainable params:** ~3.70M (L1+L2+L3+L4+PairHead). **Frozen L0:** 374k.

## Supervision

- **Teachers (mixed):** `bowwowforeach` + `Ebi`. Combined pair cache keeps acted and no-op snapshots for calibration.
- **Labels:** expert launches per turn, walked from raw replay JSON to capture full coalitions. `pair_ships (P, P) int32` carries total ships sent from each source to each target.
- **Loss:** masked BCE for `pair_logits` + source-ship-fraction MSE for `pair_frac`.

## Data flow

```
gs://orbit-wars-shipping/entity/
  code.tgz         shim + agents/transformer_v2 + scripts/build_pair_dataset_orbital_occle.py
  weights.tgz      frozen planet + fleet + comet d=256 best ckpts
  pair_cache.pt    bow+Ebi T6 mixed cache
  runs/            outputs land here as <run_name>/{entity_encoder_best.pt, log.json, ...}
```

## Guardrails baked into the cells

1. **Stale extracted code** — `extract` cell wipes `agents/`, `scripts/`, `ckpts/`, `*.pt` (except `pair_cache.pt`) and clears `sys.modules['agents.*']` + `__pycache__`.
2. **Shim asserted** — `verify_code` cell checks the minimal `agents/__init__.py` is loaded.
3. **2-head FiLM model present** — assert `m.pair_head.HEAD_NAMES == {'pair_logits', 'pair_frac'}` and `film_alpha == 1`.
4. **d_pair=256** — assert `m.d_pair == 256` (no down-projection); total param count > 3.6M.
5. **Cache present + bow+Ebi config** — cache path and model config are printed before launch.
6. **OOM guard** — default notebook batch is conservative (`BATCH_SIZE=16`) because PairHead materializes `(B,P,P,1536)` and FiLM materializes `(B,P,P,544/512)` tensors.


## 1. Authenticate + pull bundle from GCS

In [ ]:
from google.colab import auth
auth.authenticate_user()
BUCKET = 'gs://orbit-wars-shipping/entity'
print(f'pulling from {BUCKET}')

In [ ]:
import os, subprocess, time, hashlib, json, concurrent.futures
from pathlib import Path

WORK = Path('/content/orbit-wars')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

# The chunked pair cache lives at <BUCKET>/<PAIR_CACHE_PREFIX>.part_NN
# with a sibling manifest at <BUCKET>/<PAIR_CACHE_PREFIX>.manifest.json.
# Change PAIR_CACHE_PREFIX (e.g. "pair_cache" vs "pair_cache_top4") to
# select which teacher mix to train on.
PAIR_CACHE_PREFIX = 'pair_cache'

# gcloud storage cp is the modern path: it handles sliced parallel
# downloads automatically for large objects (no .boto config required)
# and parallelizes multi-object copies natively. gsutil still works
# but the newer CLI is faster and the recommended default.

def _gcs_size(url: str) -> int | None:
    """Return remote object size in bytes, or None if the object does
    not exist. Uses gcloud's machine-friendly --format output so we
    don't have to parse the Content-Length line out of a stat dump."""
    try:
        out = subprocess.run(
            ['gcloud', 'storage', 'objects', 'describe', url,
             '--format=value(size)'],
            check=True, capture_output=True, text=True,
        )
    except subprocess.CalledProcessError:
        return None
    try:
        return int(out.stdout.strip())
    except (TypeError, ValueError):
        return None

def _gcloud_cp(
    src: str, dst: Path, *,
    force: bool = False, quiet: bool = True,
) -> tuple[str, float, int]:
    name = dst.name
    if dst.exists():
        if not force:
            return name, 0.0, dst.stat().st_size
        dst.unlink()
    cmd = ['gcloud', 'storage', 'cp', src, str(dst)]
    if quiet:
        cmd.append('--quiet')
    else:
        print(f'  pulling {src}  →  {dst.name} ...', flush=True)
    t0 = time.time()
    subprocess.run(cmd, check=True)
    return name, time.time() - t0, dst.stat().st_size

# ---------------------------------------------------------------------- #
# 1) Small artifacts — code.tgz + weights.tgz                            #
# ---------------------------------------------------------------------- #
SMALL_TASKS = [
    (f'{BUCKET}/code.tgz',    WORK / 'code.tgz',    True),
    (f'{BUCKET}/weights.tgz', WORK / 'weights.tgz', True),
]

# ---------------------------------------------------------------------- #
# 2) Pair cache — chunked if a manifest exists, else single object        #
# ---------------------------------------------------------------------- #
PAIR_CACHE = WORK / 'pair_cache.pt'
MANIFEST_LOCAL = WORK / f'{PAIR_CACHE_PREFIX}.manifest.json'

def _fetch_manifest() -> dict | None:
    """Pull the chunk manifest if it exists. Returns the parsed dict or
    None (which signals "legacy single-object pair_cache.pt" mode)."""
    if _gcs_size(f'{BUCKET}/{PAIR_CACHE_PREFIX}.manifest.json') is None:
        return None
    subprocess.run(
        ['gcloud', 'storage', 'cp', '--quiet',
         f'{BUCKET}/{PAIR_CACHE_PREFIX}.manifest.json',
         str(MANIFEST_LOCAL)],
        check=True,
    )
    return json.loads(MANIFEST_LOCAL.read_text())

def _sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for blk in iter(lambda: fh.read(1 << 20), b''):
            h.update(blk)
    return h.hexdigest()

def _pull_chunk(spec: dict, chunks_dir: Path) -> tuple[str, float, int]:
    """Download one chunk into chunks_dir. Verifies sha256 if provided.
    Skips when the local chunk already matches the manifest."""
    name = spec['name']
    dst = chunks_dir / name
    expected_size = int(spec.get('size_bytes', 0))
    expected_sha = spec.get('sha256')
    if (
        dst.exists()
        and (expected_size == 0 or dst.stat().st_size == expected_size)
        and (expected_sha is None or _sha256_file(dst) == expected_sha)
    ):
        return name, 0.0, dst.stat().st_size
    if dst.exists():
        dst.unlink()
    t0 = time.time()
    print(f'    pulling chunk {name} ...', flush=True)
    subprocess.run(
        ['gcloud', 'storage', 'cp', f'{BUCKET}/{name}', str(dst)],
        check=True,
    )
    if expected_sha and _sha256_file(dst) != expected_sha:
        raise RuntimeError(f'sha256 mismatch on chunk {name}; refusing.')
    return name, time.time() - t0, dst.stat().st_size

def _assemble_chunks(manifest: dict, chunks_dir: Path) -> Path:
    """cat the chunks into pair_cache.pt; verifies total bytes match."""
    chunks = [c['name'] for c in manifest['chunks']]
    total_bytes = int(manifest.get('total_bytes', 0))
    print(f'  assembling {len(chunks)} chunks → pair_cache.pt ...')
    t0 = time.time()
    if PAIR_CACHE.exists():
        PAIR_CACHE.unlink()
    with open(PAIR_CACHE, 'wb') as out_fh:
        for name in chunks:
            with open(chunks_dir / name, 'rb') as in_fh:
                while True:
                    blk = in_fh.read(1 << 22)   # 4 MB
                    if not blk:
                        break
                    out_fh.write(blk)
    actual = PAIR_CACHE.stat().st_size
    dt = time.time() - t0
    print(f'  assembled in {dt:.1f}s  ({actual/1024**3:.2f} GB)')
    if total_bytes and actual != total_bytes:
        raise RuntimeError(
            f'assembly size mismatch: got {actual} bytes, '
            f'manifest declares {total_bytes}.'
        )
    return PAIR_CACHE

def _pull_pair_cache() -> tuple[str, float, int]:
    """Returns (label, total_wall_s, total_bytes). Picks chunked vs
    single-object based on whether the manifest object exists."""
    t0 = time.time()
    manifest = _fetch_manifest()
    if manifest is None:
        # No manifest → try the prefix-aware single object first (e.g.
        # ``pair_cache_top4.pt``), then fall back to the legacy
        # ``pair_cache.pt``. Always verify the local cache size matches
        # the remote object size before short-circuiting — a stale
        # 13 GB cache from a previous prefix must not masquerade as the
        # current request.
        for cand in (f'{PAIR_CACHE_PREFIX}.pt', 'pair_cache.pt'):
            remote_size = _gcs_size(f'{BUCKET}/{cand}')
            if remote_size is None:
                continue
            if (
                PAIR_CACHE.exists()
                and PAIR_CACHE.stat().st_size == remote_size
            ):
                return f'{cand} (single, cached)', 0.0, remote_size
            if PAIR_CACHE.exists():
                PAIR_CACHE.unlink()
            name, dt, size = _gcloud_cp(
                f'{BUCKET}/{cand}', PAIR_CACHE, force=False, quiet=False,
            )
            return f'{cand} (single)', dt, size
        raise RuntimeError(
            f'no pair cache found on {BUCKET}: tried '
            f'{PAIR_CACHE_PREFIX}.manifest.json, {PAIR_CACHE_PREFIX}.pt, '
            f'pair_cache.pt'
        )

    # Chunked path. If the assembled pair_cache.pt already matches the
    # manifest's total_bytes, skip the whole thing.
    total = int(manifest.get('total_bytes', 0))
    if (
        PAIR_CACHE.exists()
        and total
        and PAIR_CACHE.stat().st_size == total
    ):
        return f'pair_cache.pt (chunked, cached)', 0.0, PAIR_CACHE.stat().st_size

    chunks_dir = WORK / f'{PAIR_CACHE_PREFIX}_chunks'
    chunks_dir.mkdir(parents=True, exist_ok=True)
    chunk_specs = manifest['chunks']
    print(f'  chunked pair cache: {len(chunk_specs)} chunks, '
          f'{total/1024**3:.2f} GB total')
    with concurrent.futures.ThreadPoolExecutor(
        max_workers=len(chunk_specs)
    ) as pool:
        futures = {
            pool.submit(_pull_chunk, spec, chunks_dir): spec['name']
            for spec in chunk_specs
        }
        for fut in concurrent.futures.as_completed(futures):
            cname, cdt, csize = fut.result()
            mb = csize / 1024 / 1024
            if cdt == 0.0:
                print(f'    {cname:<32s} {mb:>7.1f} MB  (cached)')
            else:
                print(f'    {cname:<32s} {mb:>7.1f} MB  in {cdt:5.1f}s  '
                      f'({mb/max(cdt, 1e-3):6.1f} MB/s)')
    _assemble_chunks(manifest, chunks_dir)
    return f'pair_cache.pt (chunked, {len(chunk_specs)})', time.time() - t0, PAIR_CACHE.stat().st_size

# ---------------------------------------------------------------------- #
# 3) Run small artifacts + pair cache in parallel                         #
# ---------------------------------------------------------------------- #
T_START = time.time()
all_tasks = []
with concurrent.futures.ThreadPoolExecutor(
    max_workers=len(SMALL_TASKS) + 1
) as pool:
    futures = []
    for src, dst, force in SMALL_TASKS:
        futures.append(pool.submit(_gcloud_cp, src, dst, force=force, quiet=True))
    futures.append(pool.submit(_pull_pair_cache))
    for fut in concurrent.futures.as_completed(futures):
        all_tasks.append(fut.result())

for label, dt, size in sorted(all_tasks, key=lambda t: -t[2]):
    mb = size / 1024 / 1024
    if dt == 0.0:
        print(f'  {label:<32s} {mb:>9.1f} MB  (cached, skipped)')
    else:
        print(f'  {label:<32s} {mb:>9.1f} MB  in {dt:5.1f}s  '
              f'({mb/max(dt, 1e-3):6.1f} MB/s)')

print(f'\npair_cache.pt: {PAIR_CACHE.stat().st_size/1024**3:.2f} GB  '
      f'(total wall: {time.time()-T_START:.1f}s)')


In [ ]:
# Wipe stale extracted code so a previous bundle can't silently shadow
# the new one. The pair_cache.pt is left alone (huge file; the pull
# step's exists() check controls re-download).
!rm -rf agents scripts ckpts data
!find . -maxdepth 1 -name '*.pt' ! -name 'pair_cache.pt' -delete

!tar xzf code.tgz
!tar xzf weights.tgz

import sys
for m in list(sys.modules):
    if m.startswith('agents') or m.startswith('scripts'):
        del sys.modules[m]
import importlib, gc
importlib.invalidate_caches()
gc.collect()
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true

!ls -la

## 1b. Verify the extracted code is the latest one

Fails fast if a stale `agents/__init__.py` slipped through or the `PairHead` is missing (would indicate a pre-pair-head bundle).

In [ ]:
import sys, torch, agents
from pathlib import Path
from agents.transformer_v2.aggregator import (
    CrossEntityAttention, DualRoleAttention, JointRoleAttention, PairHead,
)
from agents.transformer_v2.pretrain.entity_encoder import EntityPretrainModel

print(f'agents module file: {agents.__file__}')
assert 'Minimal' in (agents.__doc__ or ''),     'agents/__init__.py is NOT the bundle shim — likely stale extraction; restart kernel.'

# Stack sanity — trunk + 2-head FiLM PairHead, d_pair = d_model, 8 heads everywhere
m = EntityPretrainModel(d_model=256, n_steps=6)
for attr in ('entity', 'cross', 'dual_role', 'joint_role', 'pair_head'):
    assert hasattr(m, attr), f'EntityPretrainModel missing .{attr} — stale code.tgz.'
assert not hasattr(m, 'source_decoder'), 'stale: source_decoder should be gone.'
assert not hasattr(m, 'target_decoder'), 'stale: target_decoder should be gone.'

# d_pair = d_model = 256 (no down-projection)
assert m.d_pair == m.d_model == 256, f'd_pair={m.d_pair} d_model={m.d_model} (expected 256/256)'

# 8 heads everywhere
assert m.entity.cross_attn.num_heads == 8
assert m.cross.encoder.layers[0].self_attn.num_heads == 8
assert m.dual_role.s2t_attn.num_heads == 8
assert m.dual_role.t2s_attn.num_heads == 8
assert m.joint_role.encoder.layers[0].self_attn.num_heads == 8

# Verify the 2-head FiLM API
expected_heads = {'pair_logits', 'pair_frac'}
assert set(m.pair_head.HEAD_NAMES) == expected_heads,     f'PairHead.HEAD_NAMES mismatch: {set(m.pair_head.HEAD_NAMES)} vs {expected_heads}'
assert float(m.pair_head.film_alpha.detach()) == 1.0, 'FiLM alpha must start at 1; alpha=0 deadlocks gradients.'
assert m.pair_head.pair_type_embed.num_embeddings == 27
assert m.pair_head.film_proj[0].in_features == 544
assert torch.all(m.pair_head.film_proj[-1].weight.detach() == 0), 'FiLM final weight should zero-init for identity.'
assert torch.all(m.pair_head.film_proj[-1].bias.detach() == 0), 'FiLM final bias should zero-init for identity.'

n_total = sum(p.numel() for p in m.parameters())
n_pair = sum(p.numel() for p in m.pair_head.parameters())
print(f'EntityPretrainModel total trainable: {n_total:,}  (pair_head: {n_pair:,})')
assert n_total > 3_600_000, f'expected ~3.70M params, got {n_total:,} — stale code.tgz.'

# Quick fwd shape check on random input
B, P, F, T, d = 2, 16, 64, 6, 256
planet_tokens = torch.randn(B, T, P, d)
fleet_tokens = torch.randn(B, T, F, d)
routing = {
    'fleet_target_idx': torch.randint(-1, P, (B, T, F)),
    'fleet_source_idx': torch.randint(-1, P, (B, T, F)),
    'fleet_owner_slot': torch.zeros(B, T, F, dtype=torch.long),
    'fleet_ships_log':  torch.zeros(B, T, F),
    'fleet_eta_norm':   torch.ones(B, T, F),
    'fleet_mask':       torch.ones(B, T, F, dtype=torch.bool),
}
planet_mask = torch.ones(B, T, P, dtype=torch.bool)
is_comet = torch.zeros(B, T, P, dtype=torch.bool)
out = m(planet_tokens, fleet_tokens, routing, planet_mask, is_comet=is_comet)
expected_shapes = {
    'pair_logits': (B, P, P),
    'pair_frac':   (B, P, P),
}
for name, shape in expected_shapes.items():
    assert out[name].shape == shape, f'{name}: {out[name].shape} != {shape}'
print('forward returned 2 heads with correct shapes ✓')
print('all sanity checks passed')


## 2. Stage the pair cache into the layout the pretrain CLI expects

`pretrain/entity_encoder.py`'s default `--pair-cache-path` points at `data/datasets/_pair_cache/bowwowforeach_Ebi_T6/bowwowforeach_Ebi_T6_p64_f1024_all.pt`. We hardlink (or copy) the downloaded `pair_cache.pt` into that location.

In [ ]:
from pathlib import Path
import os

# The cache subdir name encodes the player mix the cache was built from.
# pair_cache.pt (single, bow+Ebi)   → bowwowforeach_Ebi_T6
# pair_cache_top4.* (chunked, +Shun+Erfan) → bowwowforeach_Ebi_ErfanEshratifar_Shun_PI_T6
PAIR_CACHE_LAYOUT = {
    'pair_cache_top4': (
        'bowwowforeach_Ebi_ErfanEshratifar_Shun_PI_T6',
        'bowwowforeach_Ebi_ErfanEshratifar_Shun_PI_T6_p64_f1024_all.pt',
    ),
    'pair_cache': (
        'bowwowforeach_Ebi_T6',
        'bowwowforeach_Ebi_T6_p64_f1024_all.pt',
    ),
}
CACHE_SUBDIR, CACHE_FILENAME = PAIR_CACHE_LAYOUT.get(
    PAIR_CACHE_PREFIX, PAIR_CACHE_LAYOUT['pair_cache_top4'],
)

CACHE_DIR = Path(f'data/datasets/_pair_cache/{CACHE_SUBDIR}')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TARGET = CACHE_DIR / CACHE_FILENAME
SOURCE = Path('/content/orbit-wars/pair_cache.pt')

# Plain hardlink — instant on Colab's local FS.
if CACHE_TARGET.exists():
    CACHE_TARGET.unlink()
os.link(SOURCE, CACHE_TARGET)

size_gb = CACHE_TARGET.stat().st_size / 1024**3
print(f'staged: {CACHE_TARGET}')
print(f'        {size_gb:.2f} GB')

# Each player-mix has a known minimum size; bail out if the cache is
# the wrong shape entirely.
EXPECTED_MIN_GB = {
    'pair_cache_top4': 28.0,    # bow+Ebi+Shun+Erfan ≈ 32.6 GB
    'pair_cache':      10.0,    # bow+Ebi mix      ≈ 13.3 GB
}.get(PAIR_CACHE_PREFIX, 0.0)
assert size_gb >= EXPECTED_MIN_GB, (
    f'pair_cache.pt is only {size_gb:.2f} GB — expected at least '
    f'{EXPECTED_MIN_GB:.1f} GB for prefix={PAIR_CACHE_PREFIX!r}.'
)

# Store the cache path so the train cell can splice it into the CLI.
PAIR_CACHE_PATH = str(CACHE_TARGET)
print(f'PAIR_CACHE_PATH = {PAIR_CACHE_PATH}')


## 3. Verify torch is importable + GPU available

In [ ]:
import torch
print(f'torch: {torch.__version__}, cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device: {torch.cuda.get_device_name(0)}')
    print(f'  mem total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 4. Stage all three frozen L0 ckpts into run-dir layout

The pretrain CLI takes `--planet-run-dir` / `--fleet-run-dir` / `--comet-run-dir` (directories), not file paths. We arrange that here and verify each ckpt's `d_model`.

In [ ]:
import shutil
from pathlib import Path
PLANET_RUN_DIR = Path('/content/orbit-wars/ckpts/planet')
FLEET_RUN_DIR  = Path('/content/orbit-wars/ckpts/fleet')
COMET_RUN_DIR  = Path('/content/orbit-wars/ckpts/comet')
for d in (PLANET_RUN_DIR, FLEET_RUN_DIR, COMET_RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/orbit-wars/planet_encoder_best.pt', PLANET_RUN_DIR / 'planet_encoder_best.pt')
shutil.copy('/content/orbit-wars/fleet_encoder_best.pt',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt')
shutil.copy('/content/orbit-wars/comet_past_best.pt',     COMET_RUN_DIR  / 'comet_past_best.pt')

import torch
pc = torch.load(PLANET_RUN_DIR / 'planet_encoder_best.pt', map_location='cpu', weights_only=False)
fc = torch.load(FLEET_RUN_DIR  / 'fleet_encoder_best.pt',  map_location='cpu', weights_only=False)
cc = torch.load(COMET_RUN_DIR  / 'comet_past_best.pt',     map_location='cpu', weights_only=False)
print(f'planet ckpt: d_model={pc["config"]["d_model"]}, epoch={pc["epoch"]}, use_traj_branch={pc["config"].get("use_traj_branch")}')
print(f'fleet  ckpt: d_model={fc["config"]["d_model"]}, epoch={fc["epoch"]}')
print(f'comet  ckpt: d_model={cc["config"]["d_model"]}, epoch={cc["epoch"]}, input_dim={cc["config"].get("input_dim")}')

assert pc['config']['d_model'] == fc['config']['d_model'] == cc['config']['d_model'] == 256, \
    'all three L0 encoders must be d=256'

## 5. Train

The entity-encoder CLI will:

1. Read upstream `d_model` from each ckpt config and size the L0 encoders accordingly; freeze them.
2. Load `CachedPairDataset` from `--pair-cache-path` and split episodes 80/10/10 (`--val-frac 0.10 --test-frac 0.10`).
3. Per batch: stack T=6 history lazily, run L0 frozen, route via `where(is_comet, comet, planet)`, run L1 → L2 → L3 → L4 → PairHead.
4. Per epoch: `pair_logits` BCE-with-logits masked by `pair_valid`, with `pos_weight=600` on the train loss (val/test unweighted).
5. Print per-epoch table with `recall_true`, `recall_false`, `recall_at_{1, 5, 10}`.

In [ ]:
D_MODEL              = 256
D_PAIR               = 256      # PairHead projection width; None / 256 = no down-projection
ENTITY_N_HEADS       = 8        # L1 cross-attn heads (planet ← fleet)
CROSS_N_HEADS        = 8        # L2 self-attn heads per encoder layer
CROSS_N_LAYERS       = 2        # L2 number of encoder layers
DUAL_N_HEADS         = 8        # L3 + L4 heads (shared)
BATCH_SIZE           = 16       # conservative for d=256, T=6, P=64, F=1024 + FiLM
EPOCHS               = 30
LR                   = 5e-5     # conservative LR for full-width 2-head FiLM stack
WEIGHT_DECAY         = 1e-4
SEED                 = 1729
MAX_PLANETS          = 64
MAX_FLEETS           = 1024
PAIR_POS_WEIGHT      = 600.0    # pair-cell BCE imbalance (n_neg/n_pos ≈ 600)
SOURCE_ACT_POS_WEIGHT = 100.0   # legacy no-op accepted by CLI
TARGET_AIM_POS_WEIGHT = 100.0   # legacy no-op accepted by CLI
GLOB_ACT_POS_WEIGHT   = 1.0     # legacy no-op accepted by CLI
VAL_FRAC             = 0.10
TEST_FRAC            = 0.10
DEVICE               = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

import time
TS = time.strftime('%Y%m%d-%H%M%S')
OUT_DIR = f'data/runs/entity/bowEbi_pair2head_film_d{D_MODEL}_h8_lr{LR:g}_b{BATCH_SIZE}_{EPOCHS}ep_{TS}'
print('out dir:', OUT_DIR)

In [ ]:
!python -u -m agents.transformer_v2.pretrain.entity_encoder \
  --planet-run-dir $PLANET_RUN_DIR \
  --fleet-run-dir  $FLEET_RUN_DIR \
  --comet-run-dir  $COMET_RUN_DIR \
  --pair-cache-path $PAIR_CACHE_PATH \
  --out-dir $OUT_DIR \
  --d-model $D_MODEL \
  --d-pair $D_PAIR \
  --entity-n-heads $ENTITY_N_HEADS \
  --cross-n-heads $CROSS_N_HEADS \
  --cross-n-layers $CROSS_N_LAYERS \
  --dual-n-heads $DUAL_N_HEADS \
  --batch-size $BATCH_SIZE \
  --epochs $EPOCHS \
  --lr $LR \
  --weight-decay $WEIGHT_DECAY \
  --max-planets $MAX_PLANETS \
  --max-fleets $MAX_FLEETS \
  --pair-pos-weight $PAIR_POS_WEIGHT \
  --source-act-pos-weight $SOURCE_ACT_POS_WEIGHT \
  --target-aim-pos-weight $TARGET_AIM_POS_WEIGHT \
  --glob-act-pos-weight $GLOB_ACT_POS_WEIGHT \
  --val-frac $VAL_FRAC \
  --test-frac $TEST_FRAC \
  --seed $SEED \
  --device $DEVICE

## 6. Push the trained run back to GCS

In [ ]:
import subprocess
from pathlib import Path
src = Path(OUT_DIR)
assert src.is_dir(), src
# gcloud storage cp --recursive runs parallel copy across files; for
# a typical run dir (~30 MB ckpt + small logs) this is essentially
# instantaneous from Colab.
dst_parent = f'{BUCKET}/runs/'
subprocess.run(
    ['gcloud', 'storage', 'cp', '--recursive', str(src), dst_parent],
    check=True,
)
dst = f'{dst_parent}{src.name}/'
print(f'uploaded to: {dst}')
subprocess.run(
    ['gcloud', 'storage', 'ls', '--long', '--readable-sizes', dst],
    check=False,
)